# FERRUS CORPUS — corpus_main.ipynb
## Fregate 02 : Incarnation des animations R15 dans un avatar Roblox

```
IN/  ferrus_P*.fbx  +  IN_AVATAR/avatar_r15.blend
         ↓
   inject_animation.py (Blender headless)
         ↓
OUT/ corpus_P*.blend (MASTER) + corpus_P*.glb (PREVIEW)
```

**POUR L'EMPEROR. POUR LA FLOTTE FERRUS.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 0 — GIT SYNC
# Monte le Drive + clone/pull le repo + installe le codebase
# A executer EN PREMIER a chaque session Colab
# ═══════════════════════════════════════════════════════════

import os, shutil, subprocess
from google.colab import drive

# ── 0.1 Monter Google Drive ──────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ── 0.2 CONFIGURER ICI ───────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/FLOTTE-FERRUS'
GITHUB_REPO = 'https://github.com/kioka8877-ux/FLOTTE-FERRUS.git'
CLONE_DIR   = '/content/FLOTTE-FERRUS'   # clone local (ephemere, dans Colab RAM)

# ── 0.3 Clone ou Pull ────────────────────────────────────
if os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    print('[GIT SYNC] Repo deja clone — git pull...')
    proc = subprocess.run(
        ['git', '-C', CLONE_DIR, 'pull', '--rebase'],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Deja a jour.')
else:
    print('[GIT SYNC] Clone du repo...')
    proc = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_REPO, CLONE_DIR],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Clone OK.')
    if proc.returncode != 0:
        print('[GIT SYNC] ERREUR :', proc.stderr[-500:])
        raise RuntimeError('git clone echoue')

# ── 0.4 Installer le codebase sur Drive ──────────────────
SRC_CODEBASE  = os.path.join(CLONE_DIR, '02_FERRUS_CORPUS', 'codebase')
DEST_CODEBASE = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS', 'codebase')
os.makedirs(DEST_CODEBASE, exist_ok=True)

files_to_sync = ['inject_animation.py']  # fichiers du codebase a copier sur Drive
for fname in files_to_sync:
    src  = os.path.join(SRC_CODEBASE, fname)
    dest = os.path.join(DEST_CODEBASE, fname)
    shutil.copy2(src, dest)
    print(f'[GIT SYNC] Copie : {fname} → Drive/02_FERRUS_CORPUS/codebase/')

# ── 0.5 Creer les dossiers requis sur Drive ──────────────
for folder in ['IN', 'IN_AVATAR', 'OUT']:
    path = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS', folder)
    os.makedirs(path, exist_ok=True)

# ── 0.6 Bilan ────────────────────────────────────────────
print()
print('[GIT SYNC] ══════════════════════════════════════')
print('[GIT SYNC] Codebase synchronise depuis GitHub')
print(f'[GIT SYNC] DRIVE_ROOT : {DRIVE_ROOT}')
print('[GIT SYNC] Structure Drive :')
for folder in ['IN', 'IN_AVATAR', 'OUT', 'codebase']:
    path = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS', folder)
    print(f'  {"OK" if os.path.isdir(path) else "MANQUANT"}  {path}')
print('[GIT SYNC] Passer a la cellule SETUP ▶')
print('[GIT SYNC] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 1 — SETUP
# Detecte Blender, prepare les chemins (Drive deja monte en C0)
# ═══════════════════════════════════════════════════════════

import os, glob, subprocess, json

# ── 1.1 Chemins (DRIVE_ROOT defini en cellule GIT SYNC) ──
CORPUS_ROOT   = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS')
IN_DIR        = os.path.join(CORPUS_ROOT, 'IN')
IN_AVATAR_DIR = os.path.join(CORPUS_ROOT, 'IN_AVATAR')
OUT_DIR       = os.path.join(CORPUS_ROOT, 'OUT')
CODEBASE_DIR  = os.path.join(CORPUS_ROOT, 'codebase')
INJECT_SCRIPT = os.path.join(CODEBASE_DIR, 'inject_animation.py')

os.makedirs(OUT_DIR, exist_ok=True)

# ── 1.2 Detecter Blender ─────────────────────────────────
BLENDER_CANDIDATES = [
    '/content/drive/MyDrive/EXODUS_AI_MODELS/blender-4.0.0-linux-x64/blender',
    '/content/drive/MyDrive/EXODUS_AI_MODELS/BLENDER/blender-4.0.0-linux-x64/blender',
    '/usr/local/blender/blender',
    '/content/blender/blender',
]
BLENDER_BIN = next((p for p in BLENDER_CANDIDATES if os.path.isfile(p)), None)

if not BLENDER_BIN:
    print('[SETUP] Blender non trouve.')
    print('[SETUP] Renseigner manuellement : BLENDER_BIN = "/chemin/vers/blender"')
else:
    print(f'[SETUP] Blender detecte : {BLENDER_BIN}')

print(f'[SETUP] IN           : {IN_DIR}')
print(f'[SETUP] IN_AVATAR    : {IN_AVATAR_DIR}')
print(f'[SETUP] OUT          : {OUT_DIR}')
print(f'[SETUP] Script       : {"OK" if os.path.isfile(INJECT_SCRIPT) else "MANQUANT"}')
print('[SETUP] OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 2 — CONFIG
# Liste les FBX + resout l'avatar par nommage miroir
# ═══════════════════════════════════════════════════════════

# ── 2.1 Lister les FBX ───────────────────────────────────
fbx_files = sorted(glob.glob(os.path.join(IN_DIR, 'ferrus_P*.fbx')))

if not fbx_files:
    print('[CONFIG] ERREUR — Aucun ferrus_P*.fbx dans IN/')
    print(f'[CONFIG] Deposer les FBX dans : {IN_DIR}')
else:
    print(f'[CONFIG] {len(fbx_files)} FBX trouves dans IN/ :')
    for f in fbx_files:
        size_ko = os.path.getsize(f) // 1024
        print(f'         {os.path.basename(f)}  ({size_ko} Ko)')

# ── 2.2 Resolution avatar — Option A (nommage miroir) ─────
# Priorite pour ferrus_PN.fbx :
#   1. IN_AVATAR/avatar_PN.blend   (avatar dedie au personnage)
#   2. IN_AVATAR/avatar_r15.blend  (fallback commun)
#   3. IN_AVATAR/*.blend           (premier .blend disponible)

def resolve_avatar(fbx_path, in_avatar_dir):
    stem = os.path.splitext(os.path.basename(fbx_path))[0]  # ferrus_P1
    actor_id = stem.replace('ferrus_', '')                   # P1
    candidates = [
        os.path.join(in_avatar_dir, f'avatar_{actor_id}.blend'),  # avatar_P1.blend (dedie)
        os.path.join(in_avatar_dir, 'avatar_r15.blend'),           # fallback commun
    ]
    for c in candidates:
        if os.path.isfile(c):
            return c
    # Dernier recours : premier .blend disponible
    all_blends = glob.glob(os.path.join(in_avatar_dir, '*.blend'))
    return all_blends[0] if all_blends else None

print()
print('[CONFIG] Resolution des avatars (nommage miroir) :')
print(f'  {"FBX":<20} {"AVATAR":<30} {"TYPE"}')
print('  ' + '─' * 60)
avatar_map = {}
for fbx_path in fbx_files:
    stem = os.path.splitext(os.path.basename(fbx_path))[0]
    actor_id = stem.replace('ferrus_', '')
    avatar = resolve_avatar(fbx_path, IN_AVATAR_DIR)
    avatar_map[actor_id] = avatar
    if avatar:
        dedie = f'avatar_{actor_id}.blend'
        src = 'DEDIE' if os.path.basename(avatar) == dedie else 'FALLBACK'
        print(f'  {os.path.basename(fbx_path):<20} {os.path.basename(avatar):<30} {src}')
    else:
        print(f'  {os.path.basename(fbx_path):<20} {"AUCUN AVATAR TROUVE":<30} ERREUR')

# ── 2.3 Valider le script ────────────────────────────────
if not os.path.isfile(INJECT_SCRIPT):
    print(f'[CONFIG] ERREUR — inject_animation.py introuvable : {INJECT_SCRIPT}')
else:
    print(f'[CONFIG] Script OK : inject_animation.py')

# ── 2.4 Bilan ────────────────────────────────────────────
all_avatars_ok = all(v is not None for v in avatar_map.values())
ready = bool(fbx_files and all_avatars_ok and os.path.isfile(INJECT_SCRIPT) and BLENDER_BIN)
print()
print('[CONFIG] BILAN ──────────────────────────────────')
print(f'  FBX dispos   : {len(fbx_files)}')
print(f'  Avatars      : {"OK" if all_avatars_ok else "MANQUANT(S)"}')
print(f'  Script       : {"OK" if os.path.isfile(INJECT_SCRIPT) else "MANQUANT"}')
print(f'  Blender      : {"OK" if BLENDER_BIN else "MANQUANT"}')
print(f'  GO           : {"OUI" if ready else "NON — corriger les points ci-dessus"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 3 — INCARNATION
# Boucle sur N acteurs, chaque acteur avec son avatar resolu
# ═══════════════════════════════════════════════════════════

import datetime

if not ready:
    print('[INCARNATION] Conditions non remplies — executer la cellule CONFIG dabord')
    raise SystemExit

rapport_global = {
    'generated_at': datetime.datetime.now().isoformat(),
    'blender_bin': BLENDER_BIN,
    'total_actors': len(fbx_files),
    'actors': []
}

for fbx_path in fbx_files:
    stem = os.path.splitext(os.path.basename(fbx_path))[0]  # ex: ferrus_P1
    actor_id = stem.replace('ferrus_', '')                   # ex: P1
    actor_avatar = avatar_map[actor_id]                      # avatar resolu en cellule CONFIG
    out_blend  = os.path.join(OUT_DIR, f'corpus_{actor_id}.blend')
    out_glb    = os.path.join(OUT_DIR, f'corpus_{actor_id}.glb')
    report_tmp = os.path.join(OUT_DIR, f'_tmp_report_{actor_id}.json')

    print(f'[INCARNATION] ─────────────────────────────────')
    print(f'[INCARNATION] Acteur   : {actor_id}')
    print(f'[INCARNATION] FBX      : {os.path.basename(fbx_path)}')
    print(f'[INCARNATION] Avatar   : {os.path.basename(actor_avatar)}')

    if not actor_avatar:
        print(f'[INCARNATION] ERREUR — Aucun avatar pour {actor_id}, skip')
        rapport_global['actors'].append({
            'id': actor_id, 'status': 'ERREUR', 'reason': 'avatar manquant'
        })
        continue

    cmd = [
        BLENDER_BIN, '--background',
        '--python', INJECT_SCRIPT,
        '--',
        '--fbx',           fbx_path,
        '--avatar',        actor_avatar,
        '--output-blend',  out_blend,
        '--output-glb',    out_glb,
        '--report-json',   report_tmp,
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=300)

    # Afficher les logs CORPUS
    corpus_lines = [l for l in proc.stdout.splitlines() if '[CORPUS]' in l]
    for line in corpus_lines:
        print(line)

    if proc.returncode != 0:
        print(f'[INCARNATION] ERREUR Blender (code {proc.returncode})')
        error_lines = [l for l in proc.stderr.splitlines() if 'Error' in l or 'error' in l]
        for line in error_lines[-5:]:
            print(f'  {line}')
        rapport_global['actors'].append({
            'id': actor_id,
            'fbx_source': os.path.basename(fbx_path),
            'avatar_used': os.path.basename(actor_avatar),
            'status': 'ERREUR',
            'return_code': proc.returncode
        })
        continue

    # Lire le rapport individuel
    actor_report = {'id': actor_id, 'status': 'OK', 'avatar_used': os.path.basename(actor_avatar)}
    if os.path.isfile(report_tmp):
        with open(report_tmp) as f:
            actor_data = json.load(f)
        actor_report.update(actor_data)
        os.remove(report_tmp)

    rapport_global['actors'].append(actor_report)
    print(f'[INCARNATION] Acteur {actor_id} OK')

# Ecrire le rapport global
rapport_path = os.path.join(OUT_DIR, 'rapport_corpus.json')
with open(rapport_path, 'w') as f:
    json.dump(rapport_global, f, indent=2)

ok_count  = sum(1 for a in rapport_global['actors'] if a.get('status') == 'OK')
err_count = len(rapport_global['actors']) - ok_count

print()
print('[INCARNATION] ═══════════════════════════════════')
print(f'[INCARNATION] TERMINE — {ok_count}/{len(fbx_files)} acteurs incarnes')
if err_count:
    print(f'[INCARNATION] ATTENTION — {err_count} erreur(s)')
print(f'[INCARNATION] Rapport : {rapport_path}')
print('[INCARNATION] ═══════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 4 — RAPPORT
# Affiche le rapport_corpus.json
# ═══════════════════════════════════════════════════════════

rapport_path = os.path.join(OUT_DIR, 'rapport_corpus.json')

if not os.path.isfile(rapport_path):
    print('[RAPPORT] rapport_corpus.json introuvable — executer INCARNATION dabord')
else:
    with open(rapport_path) as f:
        r = json.load(f)

    print(f'[RAPPORT] Genere le : {r["generated_at"]}')
    print(f'[RAPPORT] Total acteurs : {r["total_actors"]}')
    print()
    print(f'{"ACTEUR":<10} {"BONES":<8} {"FRAMES":<8} {"BLEND (Ko)":<12} {"GLB (Ko)":<10} {"STATUT"}')
    print('─' * 65)
    for a in r['actors']:
        if a.get('status') == 'OK':
            print(
                f'{a["id"]:<10} '
                f'{a.get("bones_avatar", "?"):<8} '
                f'{a.get("frames", "?"):<8} '
                f'{a.get("output_blend_size_ko", "?"):<12} '
                f'{a.get("output_glb_size_ko", "?"):<10} '
                f'{a["status"]}'
            )
        else:
            print(f'{a["id"]:<10} {"─":<8} {"─":<8} {"─":<12} {"─":<10} ERREUR')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 5 — PREVIEW
# Verifie les fichiers OUT/ et fournit les liens viewer
# ═══════════════════════════════════════════════════════════

glb_files   = sorted(glob.glob(os.path.join(OUT_DIR, 'corpus_P*.glb')))
blend_files = sorted(glob.glob(os.path.join(OUT_DIR, 'corpus_P*.blend')))

print('[PREVIEW] Fichiers produits dans OUT/ :')
print()
print(f'  {"FICHIER":<35} {"TAILLE"}')
print('  ' + '─' * 45)

for path in blend_files + glb_files:
    size_ko = os.path.getsize(path) // 1024
    print(f'  {os.path.basename(path):<35} {size_ko} Ko')

print()
print('[PREVIEW] Pour visualiser les GLB en ligne :')
print('  → https://gltf-viewer.donmccurdy.com')
print('  → Glisser-deposer le fichier corpus_P*.glb')
print()
print('[PREVIEW] Pour utiliser dans EXODUS U01 :')
print(f'  Copier corpus_P*.blend dans :')
print(f'  DRIVE_ROOT/01_ANIMATION_ENGINE/IN_MIXAMO_BASE/')
print()
if blend_files:
    print(f'[PREVIEW] {len(blend_files)} acteur(s) prets pour production.')
else:
    print('[PREVIEW] Aucun fichier corpus produit — executer INCARNATION dabord')